In [57]:
from datasets import load_dataset


ds = load_dataset(
    "OpenAssistant/oasst1",
    split="train[:10000]"
)

Column(['6ab24d72-0181-4594-a9cd-deaf170242fb', 'c8e83833-ecbc-44fe-b6db-735228c25a1c', '6708c47f-05c9-4346-b3d2-40b2bd24fde4', '343ee2d4-87ae-41fd-a768-bdd65959dc4a', '18145bf4-37fd-4ac0-80f5-6108b5f2b365', ...])

In [39]:
## Filter dataset to get roles and language we want to work on
ds = ds.filter(
    lambda x: x['lang'] == 'en' and x['deleted'] != False
)

ds = ds.filter(
    lambda x: x["role"] in ["prompter", "assistant"]
)

Filter:   0%|          | 0/59 [00:00<?, ? examples/s]

Filter:   0%|          | 0/59 [00:00<?, ? examples/s]

In [38]:
ds

Dataset({
    features: ['message_id', 'parent_id', 'user_id', 'created_date', 'text', 'role', 'lang', 'review_count', 'review_result', 'deleted', 'rank', 'synthetic', 'model_name', 'detoxify', 'message_tree_id', 'tree_state', 'emojis', 'labels'],
    num_rows: 59
})

In [58]:
messages = {
    row["message_id"]: row
    for row in ds
}

children = {}

for row in ds:
    parent = row["parent_id"]

    if parent is not None:
        children.setdefault(parent, []).append(row["message_id"])

In [59]:
## Getting the starting point of all the messages
roots = [
    row["message_id"]
    for row in ds
    if row["parent_id"] is None
]

In [60]:
roots

['6ab24d72-0181-4594-a9cd-deaf170242fb',
 '5249c721-e6d7-45f3-9680-f211d7be308e',
 '91a934ba-cfb8-4ca9-84d0-232b43ad13ab',
 '7cce4047-8f87-42c4-9d75-a590c02be5b1',
 '2771fc62-8673-43f5-90df-be93e3eee132',
 '6dcaf26b-308f-464d-a8ab-4912502c5be2',
 '710feb81-be4c-4fde-9a3b-7fa5df007d53',
 'c866c106-86e0-4d63-97fe-03b896b0473c',
 'e480f611-0d31-433a-93d2-0e2bc675aa30',
 'c9a840de-0675-4cee-a64e-e2c49303eabd',
 'a322501a-4deb-4465-9e81-8f636b182c39',
 '1723c143-d1a2-4cb7-8cb0-4d2f3b65bc29',
 '99b7abf2-51ef-48ac-8bbd-21162e162920',
 'd5124b35-7db3-4896-83ce-edffab773013',
 'af04d707-6006-4447-a797-8134c6f13272',
 '51d8ae5b-ca9b-4343-abb5-962ab8fdc15a',
 '7f7b0c9e-d322-4ac4-b83d-2c2dfb3fe2b0',
 'fa8c92d6-6daa-42a3-a2f5-e2e0eb610c41',
 '5d2edf21-d83e-425d-9413-337f004662fc',
 'c1d38834-f3a7-4976-9e36-e5ce74174bdc',
 '400a7aba-2fee-4378-8d58-8aaeed3ca78b',
 'ca4d0c93-220b-4861-87c1-bdb44cd13f72',
 '0b8e444f-805b-4885-b7e0-262ee182870f',
 'a25bf9ca-c93d-4486-b67b-1f78ddabadbb',
 '70e9e20d-6194-

In [66]:
print("messages:", len(messages))
print("parents with children:", len(children))

messages: 10000
parents with children: 4221


In [72]:
## traversal to get all the conversation for the parent
def get_path(parent_id, conversationPath):
    conversationPath = conversationPath + [parent_id]
    if parent_id not in children:
        return [conversationPath]
    paths = []
    for child_id in children[parent_id]:
        paths.extend(get_path(child_id, conversationPath))
    return paths

In [ ]:
paths = []
for root in roots:
    paths.extend(get_path(root, []))

In [96]:
validPaths = []
for path in paths:
    if len(path)%2 == 0:
        validPaths.append(path)

In [97]:
## dataset creation for training
dataset = []

for path in validPaths:
    conversation = []

    for message_id in path:
        row = messages[message_id]

        conversation.append({
            "role": row["role"],
            "content": row["text"]
        })

    dataset.append({
        "messages": conversation
    })

Total conversations: 4951
Unique conversations: 4945
Duplicates: 6


In [ ]:
## getting all unique conversations
unique_dataset = set()

for conversation in dataset:
    messages = conversation["messages"]

    key = tuple(
        (message["role"], message["content"])
        for message in messages
    )

    unique_dataset.add(key)

print("Total conversations:", len(dataset))
print("Unique conversations:", len(unique_dataset))
print("Duplicates:", len(dataset) - len(unique_dataset))

unique_dataset = list({
    tuple((m["role"], m["content"]) for m in conv["messages"]): conv
    for conv in dataset
}.values())

print(len(unique_dataset))

Total conversations: 4951
Unique conversations: 4945
Duplicates: 6
4945


In [104]:
for conv in unique_dataset:
    for message in conv['messages']:
        if message['role'] == 'prompter':
            message['role'] = 'user'

In [105]:
unique_dataset[:1]

[{'messages': [{'role': 'user',
    'content': 'Can you write a short introduction about the relevance of the term "monopsony" in economics? Please use examples related to potential monopsonies in the labour market and cite relevant research.'},
   {'role': 'assistant',
    'content': 'Monopsony refers to a market structure where there is only one buyer of a good or service. In the context of labor markets, a monopsony occurs when there is only one employer in a particular geographic area or industry, giving them significant power to set wages and working conditions. Research has shown that the existence of monopsonies can lead to lower wages and reduced job opportunities for workers. For example, a study by the National Bureau of Economic Research found that in industries with high levels of concentration, workers earn lower wages and are less likely to receive benefits such as health insurance.'},
   {'role': 'user',
    'content': "What can be done at a regulatory level to ensure, i

In [106]:
## splitting the data in training and validation
import json
import random

random.seed(42)


split_idx = int(0.9 * len(unique_dataset))

train_dataset = unique_dataset[:split_idx]
valid_dataset = unique_dataset[split_idx:]

## save them in jsonl formate

with open("./post_training_dataset/train.jsonl", "w", encoding="utf-8") as f:
    for conversation in train_dataset:
        f.write(json.dumps(conversation, ensure_ascii=False) + "\n")

with open("./post_training_dataset/validate.jsonl", "w", encoding="utf-8") as f:
    for conversation in valid_dataset:
        f.write(json.dumps(conversation, ensure_ascii=False) + "\n")